In [ ]:
# Async send-on delta modulation

In [ ]:
import h5py as h5
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

In [ ]:
path = "../ATMEGA-AES-ASCAD_databases/ASCAD.h5" 
bd = h5.File(path, "r")

print(list(bd.keys()))

print("Keys in Profiling_traces and Attack_traces:\n")

print(list(bd["Profiling_traces"].keys()))
print(list(bd["Attack_traces"].keys()))

print("\nProfiling_traces shapes:")

print(list(bd["Profiling_traces"]["traces"].shape))
print(list(bd["Profiling_traces"]["labels"].shape))
print(list(bd["Profiling_traces"]["metadata"].shape))

print("\nAttack_traces shapes:")

print(list(bd["Attack_traces"]["traces"].shape))
print(list(bd["Attack_traces"]["labels"].shape))
print(list(bd["Attack_traces"]["metadata"].shape))

In [ ]:
plt.plot(bd["Profiling_traces"]["traces"][0])
plt.title("First Profiling Trace")
plt.xlabel("Time samples")
plt.ylabel("Amplitude")
plt.show() 

In [ ]:
import numpy as np

amostras = bd["Profiling_traces"]["traces"][:50000]

# int8 to int32 to avoid overflow
amostras_int32 = np.array(amostras, dtype=np.int32)

# derivative across time samples
derivadas = np.diff(amostras_int32, axis=1)

# convert all to absolute value
derivadas_absolutas = np.abs(derivadas)

# mean across all 10,000 traces and round to integer threshold
threshold_ideal = int(np.round(np.mean(derivadas_absolutas)))

print(f"Statistically calculated ideal threshold: {threshold_ideal}")

In [ ]:
import delta_mod

threshold = threshold_ideal
delta_traces = [list(delta_mod.delta_modulation(trace, threshold)) for trace in bd["Profiling_traces"]["traces"]]

plt.plot(delta_traces[0])
plt.title("Delta Modulated First Profiling Trace")
plt.xlabel("Time samples")
plt.ylabel("Delta Amplitude")
plt.show()

In [ ]:
rebuilt_traces = [list(delta_mod.rebuild_trace(delta_trace, bd["Profiling_traces"]["traces"][i][0], threshold)) for i, delta_trace in enumerate(delta_traces)]

plt.plot(rebuilt_traces[0])
plt.title("Rebuilt First Profiling Trace")
plt.xlabel("Time samples")
plt.ylabel("Reconstructed Amplitude")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(bd["Profiling_traces"]["traces"][0])
ax1.set_title("First Profiling Trace (Original)")
ax1.set_xlabel("Time samples")
ax1.set_ylabel("Amplitude")

ax2.plot(rebuilt_traces[0], color='orange')
ax2.set_title("Rebuilt First Profiling Trace (Delta)")
ax2.set_xlabel("Time samples")
ax2.set_ylabel("Reconstructed Amplitude")

plt.tight_layout()
plt.show()

In [ ]:
import time
import os
import re
import subprocess

trace0 = bd["Profiling_traces"]["traces"][0]

start = time.perf_counter()
_ = list(delta_mod.delta_modulation(trace0, threshold))
end = time.perf_counter()
python_time = end - start

c_time = "unavailable"
c_time_float = None

if os.path.exists("./benchmark_conversion"):
    result = subprocess.run(["./benchmark_conversion"], capture_output=True, text=True, check=False)
    match = re.search(r"([0-9]+\.[0-9]+) s", result.stdout)
    if match:
        c_time_float = float(match.group(1))
        c_time = f"{c_time_float:.9f} s"

speedup = "unavailable"
if c_time_float is not None and c_time_float > 0:
    faster_percent = ((python_time - c_time_float) / python_time) * 100
    speedup = f"{faster_percent:.2f}% faster"

print("timing comparison for delta modulation of a single trace")
print(f"time for deltamod Python: {python_time:.9f} s")
print(f"time for deltamod C: {c_time}")
print(f"C implementation is: {speedup}")

## Post Delta Mod H5 db

In [ ]:
path = "../ATMEGA-AES-ASCAD_databases/ascad_modulated.h5" 
bd = h5.File(path, "r")

print(list(bd.keys()))

print("Keys in Profiling_traces and Attack_traces:\n")

print(list(bd["Profiling_traces"].keys()))
print(list(bd["Attack_traces"].keys()))

print("\nProfiling_traces shapes:")

print(list(bd["Profiling_traces"]["traces"].shape))
print(list(bd["Profiling_traces"]["labels"].shape))
print(list(bd["Profiling_traces"]["metadata"].shape))

print("\nAttack_traces shapes:")

print(list(bd["Attack_traces"]["traces"].shape))
print(list(bd["Attack_traces"]["labels"].shape))
print(list(bd["Attack_traces"]["metadata"].shape))

In [ ]:
plt.plot(bd["Profiling_traces"]["traces"][0])
plt.title("First Profiling Trace")
plt.xlabel("Time samples")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
metadata = bd["Profiling_traces"]["metadata"]
print("metadata type:", type(metadata))
print("metadata shape:", metadata.shape)
print("metadata dtype:", metadata.dtype)
print("metadata dtype names:", metadata.dtype.names)
print("\nFirst rows:")
print(metadata[:3])
#('plaintext', 'ciphertext', 'key', 'masks', 'desync')

metadata_plaintext = metadata["plaintext"]
metadata_ciphertext = metadata["ciphertext"]
metadata_key = metadata["key"]
metadata_masks = metadata["masks"]
metadata_desync = metadata["desync"]

## Hamming Weights

In [1]:
import h5py as h5
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

from delta_mod import calculate_hw_labels
path = "../ATMEGA-AES-ASCAD_databases/ascad_modulated.h5" 
with h5.File(path, "r+") as db:
    for group in ["Profiling_traces", "Attack_traces"]:
        metadata = db[group]["metadata"]
        
        # take the first byte of plaintext and key for all traces
        pt_byte0 = metadata["plaintext"][:, 0]
        key_byte0 = metadata["key"][:, 0]
        
        hw_labels = calculate_hw_labels(pt_byte0, key_byte0)
        
        dataset_name = "labels_hw_byte0"
        
        if dataset_name in db[group]:
            del db[group][dataset_name]
            
        # save the new labels in the same group
        db[group].create_dataset(dataset_name, data=hw_labels, dtype=np.uint8)
        
        print(f"[{group}] {len(hw_labels)} calculated and saved to '{group}/{dataset_name}'")

[Profiling_traces] 50000 calculated and saved to 'Profiling_traces/labels_hw_byte0'
[Attack_traces] 10000 calculated and saved to 'Attack_traces/labels_hw_byte0'


In [ ]:
path = "../ATMEGA-AES-ASCAD_databases/ascad_modulated.h5" 
bd = h5.File(path, "r")

print(list(bd.keys()))

print("Keys in Profiling_traces and Attack_traces:\n")

print(list(bd["Profiling_traces"].keys()))
print(list(bd["Attack_traces"].keys()))

print("\nProfiling_traces shapes:")

print(list(bd["Profiling_traces"]["traces"].shape))
print(list(bd["Profiling_traces"]["labels"].shape))
print(list(bd["Profiling_traces"]["metadata"].shape))

print("\nAttack_traces shapes:")

print(list(bd["Attack_traces"]["traces"].shape))
print(list(bd["Attack_traces"]["labels"].shape))
print(list(bd["Attack_traces"]["metadata"].shape))

# hamming weight labels for the first 5 profiling traces paired with their plaintext and key bytes
for i in range(5):
    hw_label = bd["Profiling_traces"]["labels_hw_byte0"][i]
    pt_byte0 = bd["Profiling_traces"]["metadata"]["plaintext"][i][0]
    key_byte0 = bd["Profiling_traces"]["metadata"]["key"][i][0]
    print(f"Trace {i}: HW Label={hw_label}, Plaintext Byte 0={pt_byte0:#04x}, Key Byte 0={key_byte0:#04x}")

['Attack_traces', 'Profiling_traces']
Keys in Profiling_traces and Attack_traces:

['labels', 'labels_hw_byte0', 'metadata', 'traces']
['labels', 'labels_hw_byte0', 'metadata', 'traces']

Profiling_traces shapes:
[50000, 700]
[50000]
[50000]

Attack_traces shapes:
[10000, 700]
[10000]
[10000]
Trace 0: HW Label=5, Plaintext Byte 0=0x06, Key Byte 0=0x4d
Trace 1: HW Label=6, Plaintext Byte 0=0x40, Key Byte 0=0x4d
Trace 2: HW Label=4, Plaintext Byte 0=0xca, Key Byte 0=0x4d
Trace 3: HW Label=1, Plaintext Byte 0=0x31, Key Byte 0=0x4d
Trace 4: HW Label=4, Plaintext Byte 0=0x2b, Key Byte 0=0x4d
